# Alpha Construction Examples

This notebook demonstrates how the alpha complex filtration controls the interface surface.
As the radius grows, more simplices enter the complex:
- **Small radius** (r=0.51): Only edges → free edge barycenters (isolated points)
- **Medium radius** (r=0.58): Edges + triangles → free triangle and edge subdivisions (wireframe)
- **Large radius** (r=0.62): Full tetrahedron → complete surface with triangles

Examples mirror the C++ test suite (`test_alpha_construction.cpp`).

In [1]:
using JSON
using GLMakie
using GeometryBasics
using LinearAlgebra

include("../julia/src/DelaunayInterfaces.jl")
using .DelaunayInterfaces

include("../julia/src/visualization.jl")

println("DelaunayInterfaces loaded")

DelaunayInterfaces loaded


In [2]:
# Regular tetrahedron (same as C++ tests)
const TET_POINTS = [
    [0.0,  0.0,       1.0 / sqrt(3.0)],
    [0.5,  0.0,      -1.0 / (2.0 * sqrt(3.0))],
    [-0.5, 0.0,      -1.0 / (2.0 * sqrt(3.0))],
    [0.0,  sqrt(2.0 / 3.0), 0.0]
]

# Bipyramid: tetrahedron + reflected apex
const BIPYRAMID_POINTS = [
    TET_POINTS[1],
    TET_POINTS[2],
    TET_POINTS[3],
    TET_POINTS[4],
    [0.0, -sqrt(2.0 / 3.0), 0.0]
]

# Alpha radii levels
const R_EDGES = 0.51     # only edges in alpha complex
const R_TRIS  = 0.58     # edges + triangles
const R_FULL  = 0.62     # full tetrahedron

"""Compute surface and print summary."""
function alpha_summary(points, colors, r; label="")
    radii = fill(r, length(points))
    surface = InterfaceSurface(points, colors, radii; weighted=true, alpha=true)
    nv = length(surface.vertices)
    nt = count(x -> length(x[1]) == 3, surface.filtration)
    ne = count(x -> length(x[1]) == 2, surface.filtration)
    n_tets = size(surface.generating_tetrahedra, 1)
    n_ftri = length(surface.generating_free_triangles)
    n_fedg = length(surface.generating_free_edges)
    if !isempty(label)
        println("  $label: $(nv)v $(ne)e $(nt)t | mc: $(n_tets) tets, $(n_ftri) free tris, $(n_fedg) free edges")
    end
    return surface
end

println("Helpers defined")

Helpers defined


## 1. Regular Tetrahedron — Coloring Comparison

Four colorings of the same regular tetrahedron at the full alpha level (r=0.62).

In [3]:
tet_colorings = [
    (name="3-1",     colors=[1,1,1,2]),
    (name="2-1-1",   colors=[1,1,2,3]),
    (name="2-2",     colors=[1,1,2,2]),
    (name="1-1-1-1", colors=[1,2,3,4]),
]

println("Regular tetrahedron at r=$R_TRIS (full tet):")
for tc in tet_colorings
    alpha_summary(TET_POINTS, tc.colors, R_TRIS; label=tc.name)
end

Regular tetrahedron at r=0.58 (full tet):
  3-1: 6v 6e 0t | mc: 0 tets, 3 free tris, 0 free edges
  2-1-1: 9v 10e 0t | mc: 0 tets, 4 free tris, 0 free edges
  2-2: 8v 8e 0t | mc: 0 tets, 4 free tris, 0 free edges
  1-1-1-1: 10v 12e 0t | mc: 0 tets, 4 free tris, 0 free edges


In [4]:
# Visualize all four colorings side by side
fig = Figure()

for (i, tc) in enumerate(tet_colorings)
    surface = alpha_summary(TET_POINTS, tc.colors, R_FULL)
    scene = LScene(fig[1, i]; show_axis=false)
    draw_interface!(scene, surface; show_wireframe=true)

    # Draw tetrahedron edges
    pts = [Point3f(p...) for p in TET_POINTS]
    edge_pts = Point3f[]
    edge_cols = RGBA[]
    for ii in 1:4, jj in (ii+1):4
        push!(edge_pts, pts[ii], pts[jj])
        push!(edge_cols, RGBA(CONF_COLORMAP[mod1(tc.colors[ii], 4)]),
                         RGBA(CONF_COLORMAP[mod1(tc.colors[jj], 4)]))
    end
    linesegments!(scene, edge_pts; color=edge_cols, linewidth=2)

    # Draw vertices
    scatter!(scene, pts; color=[CONF_COLORMAP[mod1(c, 4)] for c in tc.colors], markersize=15)

    Label(fig[1, i, Top()], tc.name; fontsize=14)
end

display(fig)

GLMakie.Screen(...)

## 2. Alpha Filtration — Growing the Complex

For each coloring, show how the interface grows as the alpha radius increases.

In [5]:
alpha_levels = [
    (r=R_EDGES, label="r=$R_EDGES (edges)"),
    (r=R_TRIS,  label="r=$R_TRIS (edges+tris)"),
    (r=R_FULL,  label="r=$R_FULL (full tet)"),
]

for tc in tet_colorings
    println("\n$(tc.name) coloring:")
    for al in alpha_levels
        alpha_summary(TET_POINTS, tc.colors, al.r; label=al.label)
    end
end


3-1 coloring:
  r=0.51 (edges): 3v 0e 0t | mc: 0 tets, 0 free tris, 3 free edges
  r=0.58 (edges+tris): 6v 6e 0t | mc: 0 tets, 3 free tris, 0 free edges
  r=0.62 (full tet): 7v 12e 6t | mc: 1 tets, 0 free tris, 0 free edges

2-1-1 coloring:
  r=0.51 (edges): 5v 0e 0t | mc: 0 tets, 0 free tris, 5 free edges
  r=0.58 (edges+tris): 9v 10e 0t | mc: 0 tets, 4 free tris, 0 free edges
  r=0.62 (full tet): 10v 19e 10t | mc: 1 tets, 0 free tris, 0 free edges

2-2 coloring:
  r=0.51 (edges): 4v 0e 0t | mc: 0 tets, 0 free tris, 4 free edges
  r=0.58 (edges+tris): 8v 8e 0t | mc: 0 tets, 4 free tris, 0 free edges
  r=0.62 (full tet): 9v 16e 8t | mc: 1 tets, 0 free tris, 0 free edges

1-1-1-1 coloring:
  r=0.51 (edges): 6v 0e 0t | mc: 0 tets, 0 free tris, 6 free edges
  r=0.58 (edges+tris): 10v 12e 0t | mc: 0 tets, 4 free tris, 0 free edges
  r=0.62 (full tet): 11v 22e 12t | mc: 1 tets, 0 free tris, 0 free edges


In [6]:
# Visualize filtration progression for 1-1-1-1 (most interesting case)
fig = Figure(size=(1200, 400))
Label(fig[0, :], "1-1-1-1 Alpha Filtration"; fontsize=18)

colors_1111 = [1, 2, 3, 4]

for (i, al) in enumerate(alpha_levels)
    surface = alpha_summary(TET_POINTS, colors_1111, al.r)
    scene = LScene(fig[1, i]; show_axis=false)

    # Draw surface triangles + wireframe
    draw_interface!(scene, surface; show_wireframe=true)

    # Draw free simplices (edges not in any triangle, vertices not in any edge)
    draw_free_simplices!(scene, surface)

    # Draw input vertices
    pts = [Point3f(p...) for p in TET_POINTS]
    scatter!(scene, pts; color=[CONF_COLORMAP[mod1(c, 4)] for c in colors_1111], markersize=15)

    Label(fig[1, i, Top()], al.label; fontsize=13)
end

display(fig)

GLMakie.Screen(...)

In [7]:
# Visualize filtration progression for 2-1-1
fig = Figure(size=(1200, 400))
Label(fig[0, :], "2-1-1 Alpha Filtration"; fontsize=18)

colors_211 = [1, 1, 2, 3]

for (i, al) in enumerate(alpha_levels)
    surface = alpha_summary(TET_POINTS, colors_211, al.r)
    scene = LScene(fig[1, i]; show_axis=false)

    draw_interface!(scene, surface; show_wireframe=true)
    draw_free_simplices!(scene, surface)

    pts = [Point3f(p...) for p in TET_POINTS]
    scatter!(scene, pts; color=[CONF_COLORMAP[mod1(c, 4)] for c in colors_211], markersize=15)

    Label(fig[1, i, Top()], al.label; fontsize=13)
end

display(fig)

GLMakie.Screen(...)

## 3. Bipyramid Examples

A bipyramid (regular tetrahedron + reflected apex) with three colorings, showing how shared faces between tetrahedra affect the subdivision.

In [8]:
bipyramid_colorings = [
    (name="3-1 x 2 (disjoint)",      colors=[1,1,1,2,2]),
    (name="2-2 x 2 (shared face)",    colors=[1,2,2,1,1]),
    (name="1-1-1-1-1 (shared face)",  colors=[1,2,3,4,5]),
]

for bc in bipyramid_colorings
    println("\n$(bc.name):")
    for al in alpha_levels
        alpha_summary(BIPYRAMID_POINTS, bc.colors, al.r; label=al.label)
    end
end


3-1 x 2 (disjoint):
  r=0.51 (edges): 6v 0e 0t | mc: 0 tets, 0 free tris, 6 free edges
  r=0.58 (edges+tris): 12v 12e 0t | mc: 0 tets, 6 free tris, 0 free edges
  r=0.62 (full tet): 14v 24e 12t | mc: 2 tets, 0 free tris, 0 free edges

2-2 x 2 (shared face):
  r=0.51 (edges): 6v 0e 0t | mc: 0 tets, 0 free tris, 6 free edges
  r=0.58 (edges+tris): 13v 14e 0t | mc: 0 tets, 7 free tris, 0 free edges
  r=0.62 (full tet): 15v 30e 16t | mc: 2 tets, 0 free tris, 0 free edges

1-1-1-1-1 (shared face):
  r=0.51 (edges): 9v 0e 0t | mc: 0 tets, 0 free tris, 9 free edges
  r=0.58 (edges+tris): 16v 21e 0t | mc: 0 tets, 7 free tris, 0 free edges
  r=0.62 (full tet): 18v 41e 24t | mc: 2 tets, 0 free tris, 0 free edges


In [9]:
# Visualize bipyramid at full alpha (r=0.62) — all three colorings
fig = Figure(size=(1200, 400))
Label(fig[0, :], "Bipyramid at r=$R_FULL"; fontsize=18)

for (i, bc) in enumerate(bipyramid_colorings)
    surface = alpha_summary(BIPYRAMID_POINTS, bc.colors, R_FULL)
    scene = LScene(fig[1, i]; show_axis=false)

    draw_interface!(scene, surface; show_wireframe=true)
    draw_free_simplices!(scene, surface)

    pts = [Point3f(p...) for p in BIPYRAMID_POINTS]
    scatter!(scene, pts; color=[CONF_COLORMAP[mod1(c, 4)] for c in bc.colors], markersize=15)

    Label(fig[1, i, Top()], bc.name; fontsize=13)
end

display(fig)

GLMakie.Screen(...)

In [ ]:
# Filtration progression for bipyramid 1-1-1-1-1 (most complex case)
fig = Figure()
Label(fig[0, :], "Bipyramid 1-1-1-1-1 Alpha Filtration"; fontsize=18)

colors_11111 = [1, 2, 3, 4, 5]

for (i, al) in enumerate(alpha_levels)
    surface = alpha_summary(BIPYRAMID_POINTS, colors_11111, al.r)
    scene = LScene(fig[1, i]; show_axis=false)

    draw_interface!(scene, surface; show_wireframe=true)
    draw_free_simplices!(scene, surface)

    pts = [Point3f(p...) for p in BIPYRAMID_POINTS]
    scatter!(scene, pts; color=[CONF_COLORMAP[mod1(c, 4)] for c in colors_11111], markersize=15)

    Label(fig[1, i, Top()], al.label; fontsize=13)
end

display(fig)